# Stampede — a quantitative teardown 🔬
### The 12-1 decile profile · WML CAPM alpha (HAC) · the momentum-crash tail · sub-sample decay · risk-managed momentum

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Crash risk?: Severe](https://img.shields.io/badge/Crash_risk%3F-Severe-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its standard error.* The steelman is §3.1 cross-sectional momentum (Jegadeesh & Titman 1993): long winners, short losers, monthly. We prove the apparatus on a synthetic momentum panel (and a null), then read the real verdict off the S&P 500.

> ⚠️ **Not investment advice.** The core executes on synthetic data; the real run is in [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from stampede import data, momentum, strategy, decompose, extension

# Offline synthetic panel: a MOMENTUM tape (persistent relative drift -> winners keep winning) and a
# no-momentum NULL. The real S&P 500 verdict is in ../docs/results.md.
panel,  market,  truth = data.synthetic_panel(mom_strength=0.0015, seed=24)   # the momentum tape
panel0, market0, _     = data.synthetic_panel(mom_strength=0.0,    seed=24)   # the no-momentum null
print(f"{truth.n_stocks} stocks x {truth.n_bars} days | baked mom_strength={truth.mom_strength} | null=0")


150 stocks x 4032 days | baked mom_strength=0.0015 | null=0


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — do winners out-earn losers? | 🟡 `WEAK` | Overwhelming in the long-run literature and on our control (WML alpha at HAC *t* ≈ 14); but on the modern S&P 500 the WML alpha is **+4.4%/yr** (*t* = **+0.9**) — decayed to insignificance. |
| **Tradability** | 🟡 `FRAGILE` | Thin standalone Sharpe (**+0.10**), fast turnover, and you must short the losers; salvageable only with risk management. |
| **Crash risk?** | ⚪ `Severe` | Worst month **-22.5%**, max drawdown **-61%** — the loser-rebound crash. |

> **In one sentence:** the most robust anomaly in finance, real in principle and on our control, but faint on the modern large-cap sample and carrying a catastrophic, if forecastable, left tail.

*(This notebook executes on synthetic panels; the real S&P numbers are in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, precisely

Momentum score $m_{i,t} = \prod_{s=t-252}^{t-21}(1+r_{i,s}) - 1$ (the 12-1 return). WML = top-decile minus bottom-decile, equal-weight, dollar-neutral, monthly. The premium is $\mathbb{E}[r_{\text{WML}}] > 0$ with a positive CAPM alpha. The synthetic bakes a persistent per-stock relative drift $\theta_{i,t}$ (AR(1), $\phi\to1$); $\text{mom\_strength}=0$ is the null.

In [2]:
sp = momentum.momentum_spread(panel)
print(f"winners {sp['winners_ann_pct']:+.1f}%/yr vs losers {sp['losers_ann_pct']:+.1f}%/yr "
      f"-> WML spread {sp['wml_ann_pct']:+.1f}%/yr (synthetic)")

winners +26.8%/yr vs losers -1.0%/yr -> WML spread +27.8%/yr (synthetic)


## Beat 2 · So what?

Momentum's alpha is large and pervasive, but it is *conditionally* risky: its volatility and crash risk spike in panicked, post-crash markets (Daniel & Moskowitz 2016). So the open questions are: is the premium significant on the sample you can trade, how fat is the left tail, and can the tail be managed? Beats 4–6 (and the beat-7 complement) answer all three.

## Beat 3 · Pre-registered protocol

1. **Decile profile + WML alpha** (`momentum.momentum_spread`, `decompose.capm_alpha`): HAC *t*. Pre-registered: strongly positive on the momentum tape, ~0 on the null.
2. **Crash** (`decompose.crash_profile`): skew, worst months, drawdown.
3. **Decay + bootstrap** (`decompose.subsample_sharpe`, `sharpe_bootstrap`).
4. **Risk management** (`extension.crash_comparison`): vol-scaled WML.

**Verdict logic:** `WEAK`/`FRAGILE` if the modern premium is thin/insignificant and the tail is severe; the third axis records the crash.

## Beat 4 · The teardown

### 4a · WML alpha, momentum vs null

In [3]:
for label, p in [('momentum', panel), ('null', panel0)]:
    a = decompose.capm_alpha(p, cost_bps=5.0)
    print(f"{label:9s}: alpha {a['alpha_ann_pct']:+.1f}%/yr (HAC t {a['alpha_t']:+.1f}), "
          f"beta {a['beta']:+.2f}, Sharpe {a['sharpe']:+.2f}, skew {a['skew']:+.2f}")

momentum : alpha +26.4%/yr (HAC t +14.0), beta +0.05, Sharpe +3.71, skew -0.08


null     : alpha +1.4%/yr (HAC t +0.8), beta +0.06, Sharpe +0.28, skew -0.06


### 4b · The crash profile

In [4]:
cr = decompose.crash_profile(panel, cost_bps=5.0)
print(f"synthetic WML: monthly skew {cr['monthly_skew']:+.2f}, worst month {cr['worst_month_pct']:+.1f}%, "
      f"worst-5 avg {cr['worst5_months_mean_pct']:+.1f}%, max drawdown {cr['max_drawdown_pct']:.0f}%")
print('On the REAL S&P 500: worst month -22.5%, max drawdown -61% -- the loser-rebound crash.')

synthetic WML: monthly skew +0.03, worst month -3.3%, worst-5 avg -2.6%, max drawdown -6%
On the REAL S&P 500: worst month -22.5%, max drawdown -61% -- the loser-rebound crash.


> 💡 **In plain words.** Momentum's average is good but its distribution is lopsided: a few months — when the market violently reverses and the worst losers rip higher — give back years of gains. The drawdown, not the Sharpe, is what makes it hard to hold.

### 4c · Decay and bootstrap

In [5]:
display(decompose.subsample_sharpe(panel, cost_bps=5.0, n_chunks=3).round(3))
bs = decompose.sharpe_bootstrap(panel, n_boot=2000, cost_bps=5.0)
print(f"synthetic WML Sharpe {bs['sharpe']:+.2f}, 95% CI [{bs['ci_low']:+.2f}, {bs['ci_high']:+.2f}]")
print('Real S&P sub-sample Sharpe: +0.31 -> -0.40 -> +0.49 -- it comes and goes.')

,start,end,sharpe
chunk,,,
0,2009-12-22,2014-10-20,3.9860
1,2014-10-21,2019-08-19,3.7139
2,2019-08-20,2024-06-17,3.4143


synthetic WML Sharpe +3.71, 95% CI [+3.18, +4.25]
Real S&P sub-sample Sharpe: +0.31 -> -0.40 -> +0.49 -- it comes and goes.


## Beat 5 · The verdict

- **Real in principle** (4a): control WML alpha at *t* ≈ 14.
- **Faint here**: modern-S&P alpha +4.4%/yr (*t* +0.9).
- **Severe tail** (4b): worst month -22.5%, drawdown -61%.

> **Signal `WEAK` · Tradability `FRAGILE` · Crash risk? `Severe`.**

## Beat 6 · Could you trade it?

- **Thin standalone** (Sharpe +0.10) on the investable sample; you must short losers.
- **The crash** (-61% drawdown, -22.5% month) is the binding constraint.
- **Fast turnover** (7×/yr) compounds cost.

Tradability **`FRAGILE`**; the crash is `Severe` but manageable (beat 7).

## Beat 7 · Going further

### 7a · Worked complement — risk-managed momentum
The crashes cluster when the WML factor's own recent vol is high (Barroso–Santa-Clara 2015), so scale the position by inverse trailing WML vol — the [Study 16](../../16-storm-shy/) overlay — and compare.

In [6]:
cc = extension.crash_comparison(panel, cost_bps=5.0)
for k in ['plain', 'managed']:
    c = cc[k]; print(f"{k:8s}: Sharpe {c['sharpe']:+.2f}, skew {c['skew']:+.2f}, "
                     f"worst month {c['worst_month_pct']:+.1f}%, max drawdown {c['max_drawdown_pct']:.0f}%")
print('On the REAL S&P 500 (../docs/extension.md): drawdown -61% -> -32%, Sharpe +0.10 -> +0.16.')

plain   : Sharpe +3.75, skew +0.00, worst month -3.3%, max drawdown -6%
managed : Sharpe +3.74, skew -0.03, worst month -5.3%, max drawdown -9%
On the REAL S&P 500 (../docs/extension.md): drawdown -61% -> -32%, Sharpe +0.10 -> +0.16.


**The result.** Vol-scaling sidesteps the worst of the crash — on the real S&P 500 the drawdown shrinks from **-61% to -32%** and the Sharpe lifts to **+0.16** — because the tail is *forecastable* (it clusters in high-vol regimes). The same risk-management insight that earned [Study 16](../../16-storm-shy/) the desk's only green turns momentum's catastrophic tail into a merely uncomfortable one. The deeper problem stays: the *premium* is faint on the modern large-cap sample; the crash is the part you can engineer around (at the cost of leverage in calm regimes). Full run in [`../docs/extension.md`](../docs/extension.md).

### 7b · Other forks
- **Residual momentum** (§3.7) — momentum on factor-residual returns is cleaner and less crash-prone (Blitz et al.); the desk's natural next study.
- **Delisted-inclusive, all-cap history** — where the academic premium lives; date the decay.
- **Constant-volatility / dynamic-weighting momentum** (Daniel–Moskowitz) — push the crash-timing further.

PRs welcome — resurrect the premium on honest long-run data, or sharpen the crash management.